# DRB - Research Pro

Sends each Deep Research Bench query to an asynchronous research endpoint (submit, then poll for status) and appends each answer to `drb_v5_research_pro_results.jsonl` (JSONL, one `{"id", "prompt", "article"}` object per line). Goal: `<your target score>`.

Bring your own endpoint: set `BENCHMARK_API_KEY` and replace `<MODAL_WORKSPACE>` and `<MODAL_APP_NAME>` in `SUBMIT_URL` and `STATUS_URL` with your own deployment. The submit takes `{"api_key", "query"}` and returns `job_id`; the status poll takes `job_id` and `api_key` and returns `status` and, once complete, `results.report`. No data files are needed: the queries come from the pinned `deep_research_bench` clone. Research submits refuse today (`RESEARCH_COST_BOUNDED = False`).

In [ ]:
import json, os, time, requests, subprocess
from pathlib import Path

# --- Cost bound ----------------------------------------------------------------
# Spend policy: research-job submits stay off until the hosted side has a proven per-job cost ceiling
# and a working way to cancel a running job. A flat reservation per submit does not bound an accepted asynchronous
# job, and a polling timeout is not a cancellation. Status polls are not paid and stay available.
RESEARCH_COST_BOUNDED = False   # set True only after an enforceable per-job cost ceiling and cancellation on the hosted side are verified


def require_bounded_research():
    if RESEARCH_COST_BOUNDED is not True:
        raise RuntimeError(
            "Refusing to submit: a flat per-submit reservation does not bound an accepted asynchronous research job, "
            "which keeps running and billing on the hosted side after it is accepted. No hosted per-job cost ceiling "
            "or cancellation is verified, and a polling timeout is not a cancellation, so giving up on a job does "
            "not stop it. Research submits stay off until that is verified. Nothing was sent.")


# --- Spend guard -----------------------------------------------------------
# Each submit starts a paid research job on Modal. Each submit attempt (retries and failed submits included) is
# reserved at JOB_WORST_CASE_USD (default 2.00) before it is sent and settled to it after. A submit that would take
# the run past RUN_BUDGET_USD (default 25.00) is refused, and the run stops. Status polls are not paid.
# Both can be overridden with environment variables of the same name; a value that is not a finite number above 0
# stops the notebook before anything is sent. All notebooks share one ledger root and one runner.lock: the folder
# named by SPEND_LEDGER_DIR, default spend_ledger under the repo root (see _ledger_root below), always made absolute. This notebook calls no model API.
import atexit, json, math, os, threading, uuid
from datetime import datetime, timezone

# --- Canonical env parsing and ledger-root rule (identical in every notebook) ---------------------------------
# RUN_BUDGET_USD and JOB_WORST_CASE_USD: unset means the default. Any value that is set, a blank or whitespace
# value included, must parse as a finite float greater than 0, or the notebook refuses to start.
def _env_usd(name, default):
    raw = os.environ.get(name)
    if raw is None:
        return float(default)
    try:
        v = float(raw.strip())
    except ValueError:
        v = float("nan")
    if not math.isfinite(v) or v <= 0:
        raise RuntimeError(f"Refusing to start: {name} must be a finite number greater than 0 (got {raw!r}).")
    return v


RUN_BUDGET_USD = _env_usd("RUN_BUDGET_USD", 25.00)
RUN_ID = None                # None = new run with a fresh ledger. Set a non-empty string to resume and keep its spend.
JOB_WORST_CASE_USD = _env_usd("JOB_WORST_CASE_USD", 2.00)    # the most one submitted job can cost, in USD


class BudgetExceeded(RuntimeError):
    pass


# None is the only signal for a new run. Any other value is a resume ID and goes through _checked_run_id below,
# which refuses an empty string, 0, False and every other non-string value. Nothing is defaulted.
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
def _checked_run_id(value):
    """RUN_ID names one directory under SPEND_ROOT. Refuse an absolute path or any '..' segment, so the text of
    RUN_ID cannot lead os.path.join(SPEND_ROOT, RUN_ID) outside SPEND_ROOT. This is a string check only: it does
    not detect a symlink that already exists under SPEND_ROOT. Fails loudly."""
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"Refusing to start: RUN_ID must be a non-empty string (got {value!r}).")
    if os.path.isabs(value) or value.startswith(("/", "\\")) or os.path.splitdrive(value)[0]:
        raise RuntimeError(f"Refusing to start: RUN_ID must be a relative name, not an absolute path (got {value!r}).")
    if ".." in value.replace("\\", "/").split("/"):
        raise RuntimeError(f"Refusing to start: RUN_ID must not contain a '..' segment (got {value!r}).")
    return value
RUN_ID = _checked_run_id(RUN_ID)
# Ledger root. If SPEND_LEDGER_DIR is set at all, it is stripped: blank or whitespace-only refuses, anything else
# becomes os.path.abspath(os.path.expanduser(value)). If it is not set, the root is <repo root>/spend_ledger, where
# the repo root is the first directory, starting at the absolute working directory and walking up, for which
# os.path.exists(<dir>/.git) is true (a .git directory or a .git file). With no such directory the notebook refuses:
# there is no fallback to the working directory, because a different working directory would give a different lock.
# The lock is <root>/runner.lock and the run ledgers are <root>/<RUN_ID>/spend_ledger.jsonl.
def _ledger_root():
    raw = os.environ.get("SPEND_LEDGER_DIR")
    if raw is not None:
        value = raw.strip()
        if not value:
            raise RuntimeError("Refusing to start: SPEND_LEDGER_DIR is set but blank. Unset it to use "
                               "<repo root>/spend_ledger, or set it to a directory.")
        return os.path.abspath(os.path.expanduser(value))
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, ".git")):
            return os.path.join(d, "spend_ledger")
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError(f"Refusing to start: SPEND_LEDGER_DIR is not set and there is no .git above the working "
                               f"directory ({start}). Set SPEND_LEDGER_DIR to the shared ledger directory.")
        d = parent


LEDGER_ROOT = _ledger_root()
RUN_DIR = os.path.join(LEDGER_ROOT, RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)
LEDGER_PATH = os.path.join(RUN_DIR, "spend_ledger.jsonl")

# One paid runner at a time across all notebooks: one exclusive lock file in the shared ledger root. A second runner
# refuses to start while it exists. It is removed on a clean exit only. A stale lock is never cleared automatically.
RUNNER_LOCK = os.path.join(LEDGER_ROOT, "runner.lock")
try:
    _lock_fd = os.open(RUNNER_LOCK, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
except FileExistsError:
    raise RuntimeError(f"{RUNNER_LOCK} exists. Another runner may be using this ledger root, or a runner stopped without "
                       "releasing it. Refusing to start. Check for a live runner before deleting the lock by hand.")
_RUNNER_TOKEN = f"pid {os.getpid()} {datetime.now(timezone.utc).isoformat()}"
os.write(_lock_fd, _RUNNER_TOKEN.encode())
os.close(_lock_fd)


def _release_runner_lock():
    try:
        with open(RUNNER_LOCK) as f:
            if f.read() == _RUNNER_TOKEN:
                os.remove(RUNNER_LOCK)
    except OSError:
        pass


atexit.register(_release_runner_lock)

_ledger_lock = threading.Lock()
_ledger = {}                 # attempt id -> entry; reloaded from LEDGER_PATH on resume
_budget_halted = False       # set by a refused submit; no later submit is sent
if os.path.exists(LEDGER_PATH):
    with open(LEDGER_PATH) as _f:
        for _line in _f:
            if _line.strip():
                _rec = json.loads(_line)
                _ledger.setdefault(_rec["id"], {}).update(_rec)


def ledger_total():          # caller holds _ledger_lock. A reserved attempt with no settled line counts at worst case.
    return sum(e["cost"] if e["state"] == "settled" else e["worst_case"] for e in _ledger.values())


def _ledger_write(rec):
    with open(LEDGER_PATH, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def reserve_job(purpose):
    """Reserve one submit at JOB_WORST_CASE_USD before it is sent. Raises BudgetExceeded if no worst case is set,
    if the run is halted, or if the reservation would take the run past RUN_BUDGET_USD."""
    global _budget_halted
    if JOB_WORST_CASE_USD is None or not math.isfinite(JOB_WORST_CASE_USD) or JOB_WORST_CASE_USD <= 0:
        raise BudgetExceeded("not sent: JOB_WORST_CASE_USD is not a positive amount.")
    wc = round(float(JOB_WORST_CASE_USD), 6)
    with _ledger_lock:
        if _budget_halted:
            raise BudgetExceeded(f"not sent: the run is halted, so the {purpose} submit is refused")
        if ledger_total() + wc <= RUN_BUDGET_USD:
            aid = uuid.uuid4().hex
            e = {"id": aid, "state": "reserved", "purpose": purpose, "worst_case": wc,
                 "ts": datetime.now(timezone.utc).isoformat()}
            _ledger[aid] = e
            _ledger_write(e)
            return aid
        _budget_halted = True
    raise BudgetExceeded(f"not sent: {purpose} submit, worst case ${wc:.4f}, would take the run past "
                         f"RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}")


def settle_job(aid):
    """Settle one submit at its worst case. A submit is charged whether or not the job is accepted."""
    with _ledger_lock:
        e = _ledger[aid]
        e.update(state="settled", cost=e["worst_case"], cost_source="worst_case")
        _ledger_write({"id": aid, "state": "settled", "cost": e["worst_case"], "cost_source": "worst_case"})


print(f"Spend ledger: run {RUN_ID}, RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}, charged so far ${ledger_total():.4f}")

API_KEY = os.environ["BENCHMARK_API_KEY"]
SUBMIT_URL = "https://<MODAL_WORKSPACE>--<MODAL_APP_NAME>-research-pro.modal.run"
STATUS_URL  = "https://<MODAL_WORKSPACE>--<MODAL_APP_NAME>-status.modal.run"
OUTPUT_FILE = "drb_v5_research_pro_results.jsonl"
QUERY_FILE  = "deep_research_bench/data/prompt_data/query.jsonl"

print("✓ BENCHMARK_API_KEY set")

_DRB_PIN = "f2735b5c3636759c22f9e936c0232de8bf545b67"   # same commit as GRADER_REVISION in the DRB grading notebooks
if not Path("deep_research_bench").exists():
    subprocess.run(["git", "clone", "https://github.com/Ayanami0730/deep_research_bench.git"], check=True)
    subprocess.run(["git", "-C", "deep_research_bench", "checkout", "-q", "--detach", _DRB_PIN], check=True)
# Read-only check on every run, fresh clone or not. An existing checkout is never moved, reset or changed here.
_drb_head = subprocess.run(["git", "-C", "deep_research_bench", "rev-parse", "HEAD"], capture_output=True,
                           text=True, check=True).stdout.strip()
if _drb_head != _DRB_PIN:
    raise RuntimeError(f"Refusing to read the DRB queries: deep_research_bench is at {_drb_head!r}, not the pinned "
                       f"commit {_DRB_PIN}. Nothing was changed. To use the pinned commit, run by hand:\n"
                       f"  git -C deep_research_bench fetch origin {_DRB_PIN}\n"
                       f"  git -C deep_research_bench checkout --detach {_DRB_PIN}")

queries = [json.loads(l) for l in open(QUERY_FILE) if l.strip()]
print(f"Loaded {len(queries)} queries.")

completed = set()
if Path(OUTPUT_FILE).exists():
    for l in open(OUTPUT_FILE):
        try: completed.add(json.loads(l.strip())["id"])
        except: pass
    if completed: print(f"Resuming: {len(completed)} done.")

remaining = [q for q in queries if q["id"] not in completed]
print(f"{len(remaining)} remaining, ~{len(remaining)*10/60:.1f} hrs, ~${len(remaining)*JOB_WORST_CASE_USD:.0f}")

def poll_job(job_id, max_wait=1800, interval=30):
    start = time.time()
    while time.time() - start < max_wait:
        try:
            resp = requests.get(
                STATUS_URL,
                params={"job_id": job_id, "api_key": API_KEY},
                timeout=30
            )
            data = resp.json()
            status = data.get("status", "")
            elapsed = time.time() - start
            print(f"  [{elapsed:.0f}s] status: {status} | {data.get('status_message', '')}")
            if status == "complete":
                return data["results"]["report"]
            elif status in ("failed", "error"):
                print(f"  Job failed: {data.get('status_message', 'unknown')}")
                return None
        except Exception as e:
            print(f"  Poll error: {e}")
        time.sleep(interval)
    print(f"  Timed out after {max_wait}s")
    return None

require_bounded_research()

total_time, successes, failures = 0, 0, 0

for i, query in enumerate(remaining):
    require_bounded_research()
    qid    = query["id"]
    prompt = query.get("prompt", query.get("query", ""))
    print(f"\n[{i+1}/{len(remaining)}] {qid}: {prompt[:80]}...")
    article, elapsed = None, 0

    for attempt in range(1, 4):
        require_bounded_research()
        try:
            start = time.time()
            aid = reserve_job("submit")
            try:
                resp = requests.post(
                    SUBMIT_URL,
                    json={"api_key": API_KEY, "query": prompt},
                    timeout=60
                )
            finally:
                settle_job(aid)
            data = resp.json()
            if resp.status_code != 200 or data.get("error"):
                print(f"  Submit error (attempt {attempt}): {data.get('error', resp.status_code)}")
                time.sleep(60); continue
            job_id = data.get("job_id")
            if not job_id:
                print(f"  No job_id (attempt {attempt})")
                time.sleep(60); continue
            print(f"  ✓ Queued {job_id}, est. {data.get('estimated_minutes')} min — polling...")
            article = poll_job(job_id)
            elapsed = time.time() - start
            if article:
                print(f"  ✓ Complete: {len(article):,} chars, {elapsed:.0f}s")
                break
            time.sleep(60)
        except BudgetExceeded:
            raise
        except Exception as e:
            print(f"  {type(e).__name__}: {str(e)[:100]} (attempt {attempt})")
            time.sleep(60)

    if article:
        with open(OUTPUT_FILE, "a") as f:
            f.write(json.dumps({"id": qid, "prompt": prompt, "article": article}, ensure_ascii=False) + "\n")
        successes += 1; total_time += elapsed
    else:
        print(f"  ✗ FAILED: {qid}"); failures += 1

    if (i+1) % 10 == 0:
        avg = total_time / max(successes, 1)
        print(f"\n  --- {successes} done, {failures} failed, avg {avg:.0f}s/q, ETA ~{avg*(len(remaining)-i-1)/60:.0f} min ---\n")

print(f"\n{'='*50}")
print(f"  DONE: {successes} ok, {failures} failed, {total_time/60:.1f} min total")
print(f"{'='*50}")